CS 4120 Final Project

Ethan Tang

Data Cleaning and Feature Engineering

Movie reviews labeled with spoiler (0/1) at the review level.

https://www.kaggle.com/datasets/rmisra/imdb-spoiler-dataset/data?select=IMDB_reviews.json

### Data Cleaning

In [10]:
import pandas as pd

df = pd.read_json("IMDB_reviews.json", lines=True)
df

,review_date,movie_id,user_id,is_spoiler,review_text,rating,review_summary
0,10 February 2006,tt0111161,ur1898687,True,"In its Oscar year, Shawshank Redemption (writt...",10,A classic piece of unforgettable film-making.
1,6 September 2000,tt0111161,ur0842118,True,The Shawshank Redemption is without a doubt on...,10,Simply amazing. The best film of the 90's.
2,3 August 2001,tt0111161,ur1285640,True,I believe that this film is the best story eve...,8,The best story ever told on film
3,1 September 2002,tt0111161,ur1003471,True,"**Yes, there are SPOILERS here**This film has ...",10,Busy dying or busy living?
4,20 May 2004,tt0111161,ur0226855,True,At the heart of this extraordinary movie is a ...,8,"Great story, wondrously told and acted"
...,...,...,...,...,...,...,...
573908,8 August 1999,tt0139239,ur0100166,False,"Go is wise, fast and pure entertainment. Assem...",10,The best teen movie of the nineties
573909,31 July 1999,tt0139239,ur0021767,False,"Well, what shall I say. this one´s fun at any ...",9,Go - see the movie
573910,20 July 1999,tt0139239,ur0392750,False,"Go is the best movie I have ever seen, and I'v...",10,It's the best movie I've ever seen
573911,11 June 1999,tt0139239,ur0349105,False,Call this 1999 teenage version of Pulp Fiction...,3,Haven't we seen this before?


In [11]:
df = df[['movie_id', 'review_text', 'is_spoiler']]
df

,movie_id,review_text,is_spoiler
0,tt0111161,"In its Oscar year, Shawshank Redemption (writt...",True
1,tt0111161,The Shawshank Redemption is without a doubt on...,True
2,tt0111161,I believe that this film is the best story eve...,True
3,tt0111161,"**Yes, there are SPOILERS here**This film has ...",True
4,tt0111161,At the heart of this extraordinary movie is a ...,True
...,...,...,...
573908,tt0139239,"Go is wise, fast and pure entertainment. Assem...",False
573909,tt0139239,"Well, what shall I say. this one´s fun at any ...",False
573910,tt0139239,"Go is the best movie I have ever seen, and I'v...",False
573911,tt0139239,Call this 1999 teenage version of Pulp Fiction...,False


In [ ]:
# Rename to be consistent with book reviews dataset.
df = df.rename(columns={"is_spoiler": "has_spoiler"})
df

,movie_id,review_text,has_spoiler
0,tt0111161,"In its Oscar year, Shawshank Redemption (writt...",True
1,tt0111161,The Shawshank Redemption is without a doubt on...,True
2,tt0111161,I believe that this film is the best story eve...,True
3,tt0111161,"**Yes, there are SPOILERS here**This film has ...",True
4,tt0111161,At the heart of this extraordinary movie is a ...,True
...,...,...,...
573908,tt0139239,"Go is wise, fast and pure entertainment. Assem...",False
573909,tt0139239,"Well, what shall I say. this one´s fun at any ...",False
573910,tt0139239,"Go is the best movie I have ever seen, and I'v...",False
573911,tt0139239,Call this 1999 teenage version of Pulp Fiction...,False


In [ ]:
# Convert from boolean to int to be consistent with book reviews dataset.
df['has_spoiler'] = df['has_spoiler'].astype(int)
df

,movie_id,review_text,has_spoiler
0,tt0111161,"In its Oscar year, Shawshank Redemption (writt...",1
1,tt0111161,The Shawshank Redemption is without a doubt on...,1
2,tt0111161,I believe that this film is the best story eve...,1
3,tt0111161,"**Yes, there are SPOILERS here**This film has ...",1
4,tt0111161,At the heart of this extraordinary movie is a ...,1
...,...,...,...
573908,tt0139239,"Go is wise, fast and pure entertainment. Assem...",0
573909,tt0139239,"Well, what shall I say. this one´s fun at any ...",0
573910,tt0139239,"Go is the best movie I have ever seen, and I'v...",0
573911,tt0139239,Call this 1999 teenage version of Pulp Fiction...,0


In [15]:
import re

# Normalize the text.

# Apply NFKC Unicode normalization so that visually identical characters share one encoding.
df['review_text'] = df['review_text'].str.normalize('NFKC')

# Compile a pattern for genuine HTML formatting tags only, because most angle-bracket text in these reviews is written by the reviewers themselves.
HTML_TAG_RE = re.compile(r'</?(?:i|b|u|em|strong|br|p|div|span|blockquote)\s*/?>|<a\s+href=[^>]*>|</a>', flags=re.IGNORECASE)

# Replace each HTML tag with a space.
df['review_text'] = df['review_text'].str.replace(HTML_TAG_RE, ' ', regex=True)

# Compile a case-insensitive pattern for web addresses.
URL_RE = re.compile(r'https?://\S+|www\.\S+', flags=re.IGNORECASE)

# Flag the reviews that may contain a web address so that the regex replacement runs only on those rows.
has_url = df['review_text'].str.contains('http', case=False, regex=False, na=False) | df['review_text'].str.contains('www.', case=False, regex=False, na=False)

# Replace each web address with a single placeholder token.
df.loc[has_url, 'review_text'] = df.loc[has_url, 'review_text'].str.replace(URL_RE, ' URL ', regex=True)

# Split the text on any run of whitespace and rejoins it with single spaces, which also removes newlines and trims both ends.
df['review_text'] = df['review_text'].str.split().str.join(' ')

In [16]:
# Remove missing and short reviews.

# Record the row count before filtering.
n_before = len(df)

# Drop rows whose review text is missing.
df = df.dropna(subset=['review_text'])

# Set the minimum number of words that a review needs in order to carry useful signal.
MIN_WORDS = 5

# Count the whitespace-separated words in each cleaned review.
word_counts = df['review_text'].str.count(r'\S+')

# Keep only the reviews that meet the minimum word count, which also removes empty reviews.
df = df[word_counts >= MIN_WORDS]

# Report how many rows were removed and how many remain.
print(f'Removed {n_before - len(df):,} missing or short reviews; {len(df):,} remain.')

Removed 13 missing or short reviews; 573,900 remain.


In [17]:
# Remove duplicates.

# Record the row count before deduplication.
n_before = len(df)

# Remove repeated copies of the same text with the same label, keeping the first occurrence.
df = df.drop_duplicates(subset=['review_text', 'has_spoiler'], keep='first')

# Remove every copy of any text that still appears more than once, since those copies carry conflicting labels.
df = df[~df.duplicated(subset=['review_text'], keep=False)]

# Report how many rows were removed and how many remain.
print(f'Removed {n_before - len(df):,} duplicate reviews; {len(df):,} remain.')

df

Removed 641 duplicate reviews; 573,259 remain.


,movie_id,review_text,has_spoiler
1,tt0111161,The Shawshank Redemption is without a doubt on...,1
2,tt0111161,I believe that this film is the best story eve...,1
3,tt0111161,"**Yes, there are SPOILERS here**This film has ...",1
4,tt0111161,At the heart of this extraordinary movie is a ...,1
5,tt0111161,In recent years the IMDB top 250 movies has ha...,1
...,...,...,...
573908,tt0139239,"Go is wise, fast and pure entertainment. Assem...",0
573909,tt0139239,"Well, what shall I say. this one ́s fun at any...",0
573910,tt0139239,"Go is the best movie I have ever seen, and I'v...",0
573911,tt0139239,Call this 1999 teenage version of Pulp Fiction...,0


In [4]:
# Print the total number of reviews that contain a spoiler.
df['has_spoiler'].sum()

150775

In [18]:
df.to_csv('df_movie_reviews_cleaned.csv', index=False)

### Feature Engineering for Logistic Regression

In [3]:
import pandas as pd
df = pd.read_csv('df_movie_reviews_cleaned.csv')

# Set up the feature engineering steps.

# Import the os module to read the number of CPU cores.
import os

# Import NumPy for numerical arrays.
import numpy as np

# Import SciPy's sparse matrix module for the word-count and TF-IDF matrices.
import scipy.sparse as sp

# Import joblib for parallel processing and for saving the fitted vectorizer.
from joblib import Parallel, delayed, dump

# Import the scikit-learn text vectorizers.
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer, TfidfVectorizer

# Use at most eight processes so that the parallel steps stay within memory limits.
N_JOBS = min(8, os.cpu_count())


# Define a helper that applies a fixed-vocabulary vectorizer to the reviews in parallel chunks.
def transform_in_parallel(vectorizer, texts):
    # Split the texts into four chunks per process so that the work stays evenly balanced.
    chunks = np.array_split(texts.to_numpy(), N_JOBS * 4)
    # Transform every chunk in a separate process.
    parts = Parallel(n_jobs=N_JOBS)(delayed(vectorizer.transform)(chunk) for chunk in chunks)
    # Stack the chunk matrices back together in the original row order.
    return sp.vstack(parts, format='csr')

In [ ]:
# Feature Type 1: Compute TF-IDF weights for word unigrams and bigrams.

# Set the number of reviews used to choose the vocabulary, because a bigram vocabulary built from every review would need too much memory.
VOCAB_SAMPLE_SIZE = 200_000

# Configure a counter that keeps the 300,000 most frequent unigrams and bigrams that appear in at least 5 sampled reviews and in at most 90% of them.
vocab_counter = CountVectorizer(ngram_range=(1, 2), min_df=5, max_df=0.9, max_features=300_000)

# Learn the vocabulary from a reproducible random sample of reviews.
vocab_counter.fit(df['review_text'].sample(VOCAB_SAMPLE_SIZE, random_state=42))

# Keep only the learned vocabulary.
vocabulary = vocab_counter.vocabulary_

# Free the memory that the counter uses to store every discarded term.
del vocab_counter

# Configure a counter that counts only the learned vocabulary terms.
term_counter = CountVectorizer(ngram_range=(1, 2), vocabulary=vocabulary, dtype=np.float32)

# Count the vocabulary terms in every review in parallel.
term_counts = transform_in_parallel(term_counter, df['review_text'])

# Configure a transformer with sublinear term frequency, so that a word repeated many times does not dominate a review.
tfidf_transformer = TfidfTransformer(sublinear_tf=True)

# Learn the inverse document frequencies from all reviews.
tfidf_transformer.fit(term_counts)

# Convert the counts into a TF-IDF matrix with one row per row of df, without making an extra copy of the counts.
X_tfidf = tfidf_transformer.transform(term_counts, copy=False)

# Free the memory used by the raw counts.
del term_counts

# Build a TF-IDF vectorizer that converts new raw text into the same features.
tfidf = TfidfVectorizer(ngram_range=(1, 2), vocabulary=vocabulary, sublinear_tf=True, dtype=np.float32)

# Copy the learned inverse document frequencies into the vectorizer.
tfidf.idf_ = tfidf_transformer.idf_

# Save the TF-IDF matrix to disk in compressed form.
sp.save_npz('movie_X_tfidf.npz', X_tfidf, compressed=True)

# Save the fitted vectorizer to disk.
dump(tfidf, 'movie_tfidf_vectorizer.joblib')

# Display the shape of the TF-IDF matrix.
X_tfidf.shape

(573259, 300000)

In [22]:
# Feature Type 2: Measure how specific each review's words are to the reviewed movie.

# Mark which vocabulary terms are single words, because movie-specific terms such as character names are mostly unigrams.
is_unigram = np.array([' ' not in term for term in tfidf.get_feature_names_out()])

# Build a review-by-word matrix that records which vocabulary words appear in each review.
word_presence = X_tfidf[:, is_unigram]

# Replace the TF-IDF weights with ones so that the matrix records presence only.
word_presence.data[:] = 1

# Encode each movie as an integer from 0 to the number of movies minus 1.
movie_codes, movie_ids = pd.factorize(df['movie_id'])

# Store the number of movies, reviews, and vocabulary words.
n_movies, n_reviews, n_words = len(movie_ids), word_presence.shape[0], word_presence.shape[1]

# Build a movie-by-review indicator matrix that links every review to its movie.
movie_reviews = sp.csr_matrix((np.ones(n_reviews, dtype=np.float32), (movie_codes, np.arange(n_reviews))), shape=(n_movies, n_reviews))

# Count, for every movie and word, how many of the movie's reviews contain the word.
movie_word_counts = (movie_reviews @ word_presence).tocsr()

# Sort the word indices within each movie so that the movie-word pairs are in ascending order.
movie_word_counts.sort_indices()

# Count the reviews of each movie.
reviews_per_movie = np.bincount(movie_codes, minlength=n_movies)

# Count how many movies have at least one review that contains each word.
movies_per_word = np.bincount(movie_word_counts.indices, minlength=n_words)

# Compute the inverse item frequency of each word, which is high for words that appear in the reviews of only a few movies.
inverse_item_freq = np.log(n_movies / np.maximum(movies_per_word, 1)).astype(np.float32)

# Give every movie-word pair a unique integer key, which is in ascending order because the pairs are sorted.
pair_keys = np.repeat(np.arange(n_movies, dtype=np.int64), np.diff(movie_word_counts.indptr)) * n_words + movie_word_counts.indices

# Allocate an array for the mean specificity score of each review.
specificity_mean = np.zeros(n_reviews, dtype=np.float32)

# Allocate an array for the maximum specificity score of each review.
specificity_max = np.zeros(n_reviews, dtype=np.float32)

# Set the number of reviews processed at once, which limits memory use.
CHUNK_SIZE = 200_000

# Process the reviews one chunk at a time.
for start in range(0, n_reviews, CHUNK_SIZE):
    # Select the word-presence rows of the current chunk.
    chunk = word_presence[start:start + CHUNK_SIZE]
    # Find the movie of the review behind every word occurrence in the chunk.
    occurrence_movies = movie_codes[start + np.repeat(np.arange(chunk.shape[0]), np.diff(chunk.indptr))]
    # Look up how many reviews of the same movie contain each word.
    movie_counts = movie_word_counts.data[np.searchsorted(pair_keys, occurrence_movies * n_words + chunk.indices)]
    # Count the other reviews of the same movie, which excludes the review itself.
    other_reviews = reviews_per_movie[occurrence_movies] - 1
    # Compute the share of the movie's other reviews that contain each word, using zero for movies with a single review.
    other_review_share = np.divide(movie_counts - 1, other_reviews, out=np.zeros(len(movie_counts), dtype=np.float32), where=other_reviews > 0)
    # Multiply the share by the inverse item frequency to score how specific each word is to the movie.
    scores = sp.csr_matrix((other_review_share * inverse_item_freq[chunk.indices], chunk.indices, chunk.indptr), shape=chunk.shape)
    # Average the word scores within each review.
    specificity_mean[start:start + CHUNK_SIZE] = scores.sum(axis=1).A1 / np.maximum(np.diff(chunk.indptr), 1)
    # Keep the highest word score within each review.
    specificity_max[start:start + CHUNK_SIZE] = scores.max(axis=1).toarray().ravel()

# Add the mean specificity score to df.
df['movie_specificity_mean'] = specificity_mean

# Add the maximum specificity score to df.
df['movie_specificity_max'] = specificity_max

# Free the memory used by the intermediate arrays.
del word_presence, movie_reviews, movie_word_counts, pair_keys, chunk, occurrence_movies, movie_counts, other_reviews, other_review_share, scores

# Display summary statistics of the new features.
df[['movie_specificity_mean', 'movie_specificity_max']].describe()

,movie_specificity_mean,movie_specificity_max
count,573259.000000,573259.000000
mean,0.042467,1.343515
std,0.033778,0.773404
min,0.000000,0.000000
25%,0.018546,0.807219
50%,0.035355,1.327448
75%,0.057695,1.828867
max,0.832062,4.619249


In [23]:
# Feature Type 3: Compute structural and stylistic features from the review text.

# Count the words in each review, since cleaning left exactly one space between consecutive words.
word_count = df['review_text'].str.count(' ') + 1

# Count the sentences in each review as runs of end punctuation, and treat every review as having at least one sentence.
sentence_count = df['review_text'].str.count(r'[.!?]+(?:\s|$)').clip(lower=1)

# Count the capitalized words that follow a lowercase letter, comma, colon, or semicolon, which mostly captures names in the middle of a sentence.
mid_sentence_capitals = df['review_text'].str.count(r'[a-z,;:] (?=[A-Z][a-z])')

# List the first-person pronouns, which signal that the reviewer is giving an opinion.
FIRST_PERSON = ['i', 'me', 'my', 'mine', 'myself', 'we', 'us', 'our', 'ours', 'ourselves']

# List the third-person pronouns, which signal that the reviewer is retelling the plot.
THIRD_PERSON = ['he', 'him', 'his', 'himself', 'she', 'her', 'hers', 'herself', 'they', 'them', 'their', 'theirs', 'themselves']

# Configure a counter for the pronouns that also keeps one-letter words such as "i".
pronoun_counter = CountVectorizer(vocabulary=FIRST_PERSON + THIRD_PERSON, token_pattern=r'(?u)\b\w+\b', dtype=np.float32)

# Count the pronouns in every review in parallel.
pronoun_counts = transform_in_parallel(pronoun_counter, df['review_text'])

# Add up the first-person pronouns in each review.
first_person_count = pronoun_counts[:, :len(FIRST_PERSON)].sum(axis=1).A1

# Add up the third-person pronouns in each review.
third_person_count = pronoun_counts[:, len(FIRST_PERSON):].sum(axis=1).A1

# Add the log-scaled word count to df, because review lengths are heavily skewed.
df['log_word_count'] = np.log1p(word_count).astype(np.float32)

# Add the log-scaled sentence count to df.
df['log_sentence_count'] = np.log1p(sentence_count).astype(np.float32)

# Add the log-scaled average number of words per sentence to df.
df['log_avg_sentence_length'] = np.log1p(word_count / sentence_count).astype(np.float32)

# Add the share of words that are capitalized in the middle of a sentence to df.
df['capitalized_word_density'] = (mid_sentence_capitals / word_count).astype(np.float32)

# Add the log ratio of third-person to first-person pronouns to df, adding one to each count to avoid division by zero.
df['log_third_first_pronoun_ratio'] = np.log((third_person_count + 1) / (first_person_count + 1)).astype(np.float32)

# Add the log-scaled count of quotation marks to df.
df['log_quote_count'] = np.log1p(df['review_text'].str.count('"')).astype(np.float32)

# Add the log-scaled count of exclamation marks to df.
df['log_exclamation_count'] = np.log1p(df['review_text'].str.count('!')).astype(np.float32)

# Add the log-scaled count of question marks to df.
df['log_question_count'] = np.log1p(df['review_text'].str.count(r'\?')).astype(np.float32)

# Add the log-scaled count of numbers to df.
df['log_number_count'] = np.log1p(df['review_text'].str.count(r'\d+')).astype(np.float32)

# Free the memory used by the intermediate results.
del word_count, sentence_count, mid_sentence_capitals, pronoun_counts, first_person_count, third_person_count

# Display summary statistics of the new features.
df.loc[:, 'log_word_count':].describe()

,log_word_count,log_sentence_count,log_avg_sentence_length,capitalized_word_density,log_third_first_pronoun_ratio,log_quote_count,log_exclamation_count,log_question_count,log_number_count
count,573259.000000,573259.000000,573259.000000,573259.000000,573259.000000,573259.000000,573259.000000,573259.000000,573259.000000
mean,5.328053,2.373106,3.126322,0.054010,-0.203245,0.671162,0.363691,0.308631,0.737334
std,0.674513,0.639875,0.485039,0.045670,1.124410,0.905811,0.618306,0.543500,0.734549
min,1.791759,0.693147,0.723128,0.000000,-4.691348,0.000000,0.000000,0.000000,0.000000
25%,4.882802,1.945910,2.819546,0.023077,-0.998529,0.000000,0.000000,0.000000,0.000000
50%,5.247024,2.397895,3.069602,0.046053,-0.223144,0.000000,0.000000,0.000000,0.693147
75%,5.774551,2.772589,3.348734,0.074656,0.510826,1.098612,0.693147,0.693147,1.386294
max,7.892078,5.308268,6.889591,0.933333,4.430817,4.997212,6.293419,4.574711,5.517453


In [24]:
# Feature Type 4: Spoiler keyword count and spoiler keyword flag.

# List cue words about death.
DEATH_CUES = ['die', 'dies', 'died', 'dying', 'death', 'dead', 'kill', 'kills', 'killed', 'killing', 'killer', 'murder', 'murdered', 'murderer']

# List cue words and phrases about endings.
ENDING_CUES = ['ending', 'ended', 'ends up', 'ended up', 'the end', 'epilogue', 'finale', 'last chapter', 'final chapter', 'last page']

# List cue words and phrases about twists and revelations.
TWIST_CUES = ['twist', 'twists', 'reveal', 'reveals', 'revealed', 'revelation', 'turns out', 'turned out', 'finds out', 'found out', 'cliffhanger', 'betray', 'betrays', 'betrayed', 'betrayal']

# List cue words about major life events of the characters.
LIFE_EVENT_CUES = ['married', 'marries', 'marry', 'wedding', 'pregnant', 'pregnancy', 'survive', 'survives', 'survived', 'sacrifice', 'sacrificed']

# List the words that mention spoilers directly.
SPOILER_MENTIONS = ['spoiler', 'spoilers']

# List the phrases that state that a review avoids spoilers, which are subtracted from the count.
NEGATED_MENTIONS = ['spoiler free', 'spoilers free', 'no spoiler', 'no spoilers']

# Combine the cue lists into one list of keywords.
SPOILER_KEYWORDS = DEATH_CUES + ENDING_CUES + TWIST_CUES + LIFE_EVENT_CUES + SPOILER_MENTIONS

# Configure a counter for the keywords and the negated phrases.
keyword_counter = CountVectorizer(ngram_range=(1, 2), vocabulary=SPOILER_KEYWORDS + NEGATED_MENTIONS, dtype=np.float32)

# Count the keywords and negated phrases in every review in parallel.
keyword_counts = transform_in_parallel(keyword_counter, df['review_text'])

# Add up the keyword occurrences in each review.
cue_count = keyword_counts[:, :len(SPOILER_KEYWORDS)].sum(axis=1).A1

# Add up the negated phrases in each review.
negated_count = keyword_counts[:, len(SPOILER_KEYWORDS):].sum(axis=1).A1

# Detect the "** spoiler alert **" header that Goodreads adds to the start of some reviews, because the reviewer did not write it.
has_alert_header = df['review_text'].str.match(r'\*\* spoiler alert \*\*', case=False).to_numpy()

# Add the keyword count to df, removing one mention of "spoiler" for each negated phrase and for the header, and never going below zero.
df['spoiler_keyword_count'] = np.clip(cue_count - negated_count - has_alert_header, 0, None).astype(np.float32)

# Add a flag to df that marks whether a review contains at least one keyword.
df['spoiler_keyword_flag'] = (df['spoiler_keyword_count'] > 0).astype(np.int8)

# Free the memory used by the intermediate results.
del keyword_counts, cue_count, negated_count, has_alert_header

# Display the spoiler rate of the reviews with and without a keyword.
df.groupby('spoiler_keyword_flag')['has_spoiler'].agg(['count', 'mean'])

,count,mean
spoiler_keyword_flag,,
0,289406,0.162094
1,283853,0.365908


In [25]:
df

,movie_id,review_text,has_spoiler,movie_specificity_mean,movie_specificity_max,log_word_count,log_sentence_count,log_avg_sentence_length,capitalized_word_density,log_third_first_pronoun_ratio,log_quote_count,log_exclamation_count,log_question_count,log_number_count,spoiler_keyword_count,spoiler_keyword_flag
1,tt0111161,The Shawshank Redemption is without a doubt on...,1,0.049105,1.054192,5.411646,2.197225,3.362976,0.116592,0.405465,1.098612,0.000000,0.000000,1.791759,1.0,1
2,tt0111161,I believe that this film is the best story eve...,1,0.040800,1.054192,5.525453,2.708050,2.936892,0.044000,0.287682,1.609438,0.000000,0.000000,0.693147,2.0,1
3,tt0111161,"**Yes, there are SPOILERS here**This film has ...",1,0.007063,0.561976,6.609349,3.526361,3.155064,0.013495,0.424883,0.000000,0.000000,0.000000,0.000000,3.0,1
4,tt0111161,At the heart of this extraordinary movie is a ...,1,0.024900,1.054192,6.721426,3.433987,3.354572,0.098914,0.826679,2.564949,1.098612,0.000000,2.197225,4.0,1
5,tt0111161,In recent years the IMDB top 250 movies has ha...,1,0.028315,1.054192,6.167517,3.044523,3.210844,0.050420,-0.105361,0.000000,2.564949,1.609438,2.079442,4.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
573908,tt0139239,"Go is wise, fast and pure entertainment. Assem...",0,0.047975,1.326467,3.970292,1.609438,2.639057,0.057692,-1.098612,0.000000,0.000000,0.000000,0.000000,0.0,0
573909,tt0139239,"Well, what shall I say. this one ́s fun at any...",0,0.001039,0.011000,4.025352,2.079442,2.181224,0.000000,-1.098612,0.000000,0.693147,0.000000,0.000000,0.0,0
573910,tt0139239,"Go is the best movie I have ever seen, and I'v...",0,0.016142,0.534940,5.313206,2.944439,2.503256,0.044554,-0.980829,2.944439,0.000000,0.000000,0.693147,0.0,0
573911,tt0139239,Call this 1999 teenage version of Pulp Fiction...,0,0.012389,0.364500,4.499810,1.945910,2.762117,0.033708,-1.386294,0.000000,0.000000,0.000000,1.386294,0.0,0


In [26]:
df.to_csv('df_movie_reviews_spoiler_features.csv', index=False)